In [1]:
# ==========================================================
# Future Imports
# ==========================================================
from __future__ import annotations

# ==========================================================
# Standard Library
# ==========================================================
import os
import json
import shutil
import logging
import tempfile
from pathlib import Path
from typing import Any, Dict, List, Optional

# ==========================================================
# Data Science
# ==========================================================
import numpy as np
import pandas as pd

# ==========================================================
# Visualization (optional)
# ==========================================================
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================================
# Progress Bar
# ==========================================================
from tqdm import tqdm

# ==========================================================
# Environment Variables
# ==========================================================
from dotenv import load_dotenv

# ==========================================================
# OCR & Image Processing
# ==========================================================
import cv2
import pytesseract
from pytesseract import Output
from PIL import Image

# ==========================================================
# PDF Parsing
# ==========================================================
from pypdf import PdfReader
import pdfplumber
import pypdfium2

# ==========================================================
# DOCX Parsing
# ==========================================================
import docx2txt

try:
    from docx2pdf import convert as docx2pdf_convert
except ImportError:
    docx2pdf_convert = None

# ==========================================================
# LlamaIndex Core
# ==========================================================
from llama_index.core import (
    Document,
    Settings,
    StorageContext,
    VectorStoreIndex,
    SimpleDirectoryReader,
)

# ==========================================================
# Node Parsers (Chunking)
# ==========================================================
from llama_index.core.node_parser import (
    SentenceSplitter,
    TokenTextSplitter,
    SemanticSplitterNodeParser,
    MarkdownNodeParser,
)

# ==========================================================
# Embeddings
# ==========================================================
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# ==========================================================
# LLM
# ==========================================================
from llama_index.llms.ollama import Ollama

# ==========================================================
# Vector Store
# ==========================================================
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

# ==========================================================
# BM25 (Hybrid Search)
# ==========================================================
from rank_bm25 import BM25Okapi

# ==========================================================
# FastAPI
# ==========================================================
from fastapi import (
    FastAPI,
    UploadFile,
    File,
    Form,
    HTTPException,
)

from fastapi.responses import JSONResponse

# ==========================================================
# Streamlit
# ==========================================================
import streamlit as st

# ==========================================================
# Machine Learning
# ==========================================================
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================================
# Utilities
# ==========================================================
import requests
import httpx

c:\Users\sandi\Desktop\ML Working Folder\hireintel_ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
current_dir = Path.cwd()
print(f"The current working directory is : {current_dir}")

project_root = current_dir.parent
print(f"The current working directory is : {project_root}")

original_resume_folder_path = project_root / "data" / "original"
print(f"The current working directory is : {original_resume_folder_path}")

processed_resume_folder_path = project_root / "data" / "processed"
print(f"The current working directory is : {processed_resume_folder_path}")

The current working directory is : c:\Users\sandi\Desktop\ML Working Folder\hireintel_ai\notebooks
The current working directory is : c:\Users\sandi\Desktop\ML Working Folder\hireintel_ai
The current working directory is : c:\Users\sandi\Desktop\ML Working Folder\hireintel_ai\data\original
The current working directory is : c:\Users\sandi\Desktop\ML Working Folder\hireintel_ai\data\processed


# Chunking Analysis :

Step 1. Load the PDF

In [3]:
from pathlib import Path
from llama_index.core import SimpleDirectoryReader

pdf_path = original_resume_folder_path / "BusinessAnalyst" / "0d0d565220bd4a5c.pdf"

print("=" * 80)
print("PDF PATH")
print("=" * 80)
print(pdf_path)

print("\n" + "=" * 80)
print("DOES THE FILE EXIST?")
print("=" * 80)
print(pdf_path.exists())

documents = SimpleDirectoryReader(
    input_files=[str(pdf_path)]
).load_data()

print("\n" + "=" * 80)
print("DOCUMENT OBJECT TYPE")
print("=" * 80)
print(type(documents[0]))

print("\n" + "=" * 80)
print("DOCUMENT METADATA")
print("=" * 80)
print(documents[0].metadata)

print("\n" + "=" * 80)
print("FIRST 1000 CHARACTERS OF THE DOCUMENT")
print("=" * 80)
print(documents[0].text[:1000])

print("\n" + "=" * 80)
print("TOTAL NUMBER OF CHARACTERS")
print("=" * 80)
print(len(documents[0].text))

PDF PATH
c:\Users\sandi\Desktop\ML Working Folder\hireintel_ai\data\original\BusinessAnalyst\0d0d565220bd4a5c.pdf

DOES THE FILE EXIST?
True

DOCUMENT OBJECT TYPE
<class 'llama_index.core.schema.Document'>

DOCUMENT METADATA
{'page_label': '1', 'file_name': '0d0d565220bd4a5c.pdf', 'file_path': 'c:\\Users\\sandi\\Desktop\\ML Working Folder\\hireintel_ai\\data\\original\\BusinessAnalyst\\0d0d565220bd4a5c.pdf', 'file_type': 'application/pdf', 'file_size': 261613, 'creation_date': '2026-06-18', 'last_modified_date': '2026-06-18'}

FIRST 1000 CHARACTERS OF THE DOCUMENT
Details
Address
New York, NY
United States
Phone
917-678-5454
Email
kaitlyn_rianalyst@gmail.com
Skills
Business Development and
Management
Leadership
@eeee
Problem Solving
@e@eee0e
Attention to Detail
@eeee
Interpersonal Skills
@eeee
Finance and Accounting
@eeee
Languages
English
@eeeee0
Spanish; Castilian
@eee
French
@eoee
Kaitlyn Riley
Business
Profile
Dedicated Business Analyst with a strong understanding of finance, opera

Sentence Splitter Chunking :

In [4]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(
    chunk_size=300,
    chunk_overlap=100
)

nodes = splitter.get_nodes_from_documents(documents)

print(f"Total chunks: {len(nodes)}")

Total chunks: 2


In [5]:
for i, node in enumerate(nodes):

    print("=" * 80)
    print(f"Chunk {i+1}")
    print("=" * 80)

    print(node.text)

    print("\nCharacter Count :", len(node.text))

Chunk 1
Details
Address
New York, NY
United States
Phone
917-678-5454
Email
kaitlyn_rianalyst@gmail.com
Skills
Business Development and
Management
Leadership
@eeee
Problem Solving
@e@eee0e
Attention to Detail
@eeee
Interpersonal Skills
@eeee
Finance and Accounting
@eeee
Languages
English
@eeeee0
Spanish; Castilian
@eee
French
@eoee
Kaitlyn Riley
Business
Profile
Dedicated Business Analyst with a strong understanding of finance, operational
proficiency, and business development. Adept in project management and trend analysis.
Committed to gaining extensive knowledge of the structure, policies, and operations of an
organization, to properly recommend solutions that improve business processes.
Employment History
Business Analyst, Lexagon
Jun 2019 — Present New York
e Steadily provide direct support to process improvement in customer service
operations.
e Analyze, define, and develop complex business process methods to simplify and
improve the customer experience for this busy recruiting f

Document Aware Chunking :

In [19]:
# ==========================================================
# STEP 2: DOCUMENT-AWARE CHUNKING
# ==========================================================

from docling.document_converter import DocumentConverter
from llama_index.core import Document
from llama_index.core.node_parser import MarkdownNodeParser

# ----------------------------------------------------------
# 1. Convert PDF to Markdown
# ----------------------------------------------------------

print("=" * 80)
print("CONVERTING PDF TO MARKDOWN")
print("=" * 80)

converter = DocumentConverter()

result = converter.convert(str(pdf_path))

markdown_text = result.document.export_to_markdown()

# ----------------------------------------------------------
# 2. Preview the Markdown
# ----------------------------------------------------------

print("\n" + "=" * 80)
print("MARKDOWN PREVIEW (First 2000 Characters)")
print("=" * 80)

print(markdown_text[:2000])

print("\n" + "=" * 80)
print("TOTAL MARKDOWN LENGTH")
print("=" * 80)

print(len(markdown_text))

# ----------------------------------------------------------
# 3. Convert Markdown to LlamaIndex Document
# ----------------------------------------------------------

markdown_document = Document(text=markdown_text)

# ----------------------------------------------------------
# 4. Apply Document-Aware Chunking
# ----------------------------------------------------------

parser = MarkdownNodeParser()

nodes = parser.get_nodes_from_documents([markdown_document])

# ----------------------------------------------------------
# 5. View Summary
# ----------------------------------------------------------

print("\n" + "=" * 80)
print("DOCUMENT-AWARE CHUNKING COMPLETE")
print("=" * 80)

print(f"Total Chunks Created : {len(nodes)}")

# ----------------------------------------------------------
# 6. Display Every Chunk
# ----------------------------------------------------------

for i, node in enumerate(nodes):

    print("\n" + "=" * 80)
    print(f"CHUNK {i+1}")
    print("=" * 80)

    print(f"Characters : {len(node.text)}")

    print("\nMetadata:")
    print(node.metadata)

    print("\nText:")
    print("-" * 80)
    print(node.text)

2026-07-04 14:07:08,953 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-07-04 14:07:09,073 - INFO - Going to convert document batch...
2026-07-04 14:07:09,074 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 4f039a2d03e2feb47a86efdc087cc2c6
2026-07-04 14:07:09,133 - INFO - Loading plugin 'docling_defaults'


CONVERTING PDF TO MARKDOWN


2026-07-04 14:07:09,141 - INFO - Registered picture descriptions: ['picture_description_vlm_engine', 'vlm', 'api']
2026-07-04 14:07:09,198 - INFO - Loading plugin 'docling_defaults'
2026-07-04 14:07:09,225 - INFO - Registered ocr engines: ['auto', 'easyocr', 'kserve_v2_ocr', 'nemotron-ocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2026-07-04 14:07:09,667 - INFO - Accelerator device: 'cpu'
[INFO] 2026-07-04 14:07:09,700 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-04 14:07:09,718 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\sandi\Desktop\ML Working Folder\hireintel_ai\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-04 14:07:09,720 [RapidOCR] main.py:63: Using C:\Users\sandi\Desktop\ML Working Folder\hireintel_ai\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-04 14:07:09,822 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-04 14:07:09,826 [RapidOCR] down


MARKDOWN PREVIEW (First 2000 Characters)
Details

Address

New York, NY

United States

Phone

917-678-5454

Email

kaitlyn\_rianalyst@gmail.com

Skil s

Business Development and

Management

Leadership

@eeee

Problem Solving

@e@eee0e

At ention to Detail

@eeee

Interpersonal Skil s

@eeee

Finance and Accounting

@eeee

Languages

English

@eeeee0

Spanish; Castil an

@eee

French

@eoee

Kaitlyn Riley

Business

Profile

Dedicated Business Analyst with a strong understanding of finance, operational

proficiency, and business development. Adept in project management and trend analysis.

Commit ed to gaining extensive knowledge of the structure, policies, and operations of an

organization, to properly recommend solutions that improve business processes.

Employment History

Business Analyst, Lexagon Business Development and Management

Jun 2019 - Present New York

e Steadily provide direct support to process improvement in customer service

operations.

e Analyze, define, and deve

Semantic Chunking :

In [20]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

2026-07-04 14:17:34,742 - INFO - No device provided, using cpu
2026-07-04 14:17:35,014 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-04 14:17:35,029 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-07-04 14:17:35,273 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-04 14:17:35,287 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-04 14:17:35,297 - INFO - Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
2026-07-04 14

In [22]:
# ==========================================================
# STEP 3 : SEMANTIC CHUNKING
# ==========================================================

from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SemanticSplitterNodeParser

# ----------------------------------------------------------
# 1. Load Embedding Model
# ----------------------------------------------------------

print("=" * 80)
print("STEP 3.1 : LOADING EMBEDDING MODEL")
print("=" * 80)

embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded Successfully!")

# ----------------------------------------------------------
# 2. Create Semantic Splitter
# ----------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 3.2 : CREATING SEMANTIC SPLITTER")
print("=" * 80)

semantic_splitter = SemanticSplitterNodeParser(
    embed_model=embed_model,
    breakpoint_percentile_threshold=95,
)

print("Semantic Splitter Created Successfully!")

# ----------------------------------------------------------
# 3. Chunk the Document
# ----------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 3.3 : GENERATING SEMANTIC CHUNKS")
print("=" * 80)

nodes = semantic_splitter.get_nodes_from_documents(documents)

# ----------------------------------------------------------
# 4. Chunk Summary
# ----------------------------------------------------------

print("\n" + "=" * 80)
print("SEMANTIC CHUNK SUMMARY")
print("=" * 80)

print(f"Total Chunks Created : {len(nodes)}\n")

for i, node in enumerate(nodes):

    print(
        f"Chunk {i+1:2} | "
        f"Characters : {len(node.text):4} | "
        f"Words : {len(node.text.split()):4}"
    )

# ----------------------------------------------------------
# 5. Display Every Chunk
# ----------------------------------------------------------

for i, node in enumerate(nodes):

    print("\n" + "=" * 100)
    print(f"CHUNK {i+1}")
    print("=" * 100)

    print(f"Characters : {len(node.text)}")
    print(f"Words      : {len(node.text.split())}")

    print("\nMetadata")
    print("-" * 100)

    if node.metadata:
        for key, value in node.metadata.items():
            print(f"{key}: {value}")
    else:
        print("No Metadata")

    print("\nChunk Text")
    print("-" * 100)

    print(node.text)

print("\n" + "=" * 100)
print("END OF SEMANTIC CHUNKING")
print("=" * 100)

STEP 3.1 : LOADING EMBEDDING MODEL


2026-07-04 14:21:08,213 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-04 14:21:08,230 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-07-04 14:21:08,472 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-07-04 14:21:08,718 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-04 14:21:08,733 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.j

Embedding Model Loaded Successfully!

STEP 3.2 : CREATING SEMANTIC SPLITTER
Semantic Splitter Created Successfully!

STEP 3.3 : GENERATING SEMANTIC CHUNKS

SEMANTIC CHUNK SUMMARY
Total Chunks Created : 2

Chunk  1 | Characters :  523 | Words :   65
Chunk  2 | Characters : 1265 | Words :  188

CHUNK 1
Characters : 523
Words      : 65

Metadata
----------------------------------------------------------------------------------------------------
page_label: 1
file_name: 0d0d565220bd4a5c.pdf
file_path: c:\Users\sandi\Desktop\ML Working Folder\hireintel_ai\data\original\BusinessAnalyst\0d0d565220bd4a5c.pdf
file_type: application/pdf
file_size: 261613
creation_date: 2026-06-18
last_modified_date: 2026-06-18

Chunk Text
----------------------------------------------------------------------------------------------------
Details
Address
New York, NY
United States
Phone
917-678-5454
Email
kaitlyn_rianalyst@gmail.com
Skills
Business Development and
Management
Leadership
@eeee
Problem Solving
@e@ee

In [25]:
# ==========================================================
# STEP 4 : RECURSIVE CHUNKING (LangChain)
# ==========================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

# ----------------------------------------------------------
# 1. Create Recursive Splitter
# ----------------------------------------------------------

print("=" * 80)
print("STEP 4.1 : CREATING RECURSIVE SPLITTER")
print("=" * 80)

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

print("Recursive Splitter Created Successfully!")

# ----------------------------------------------------------
# 2. Split the Resume
# ----------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 4.2 : GENERATING RECURSIVE CHUNKS")
print("=" * 80)

chunks = recursive_splitter.split_text(documents[0].text)

# ----------------------------------------------------------
# 3. Chunk Summary
# ----------------------------------------------------------

print("\n" + "=" * 80)
print("RECURSIVE CHUNK SUMMARY")
print("=" * 80)

print(f"Total Chunks Created : {len(chunks)}\n")

for i, chunk in enumerate(chunks):

    print(
        f"Chunk {i+1:2} | "
        f"Characters : {len(chunk):4} | "
        f"Words : {len(chunk.split()):4}"
    )

# ----------------------------------------------------------
# 4. Display Every Chunk
# ----------------------------------------------------------

for i, chunk in enumerate(chunks):

    print("\n" + "=" * 100)
    print(f"CHUNK {i+1}")
    print("=" * 100)

    print(f"Characters : {len(chunk)}")
    print(f"Words      : {len(chunk.split())}")

    print("\nChunk Text")
    print("-" * 100)

    print(chunk)

print("\n" + "=" * 100)
print("END OF RECURSIVE CHUNKING")
print("=" * 100)

STEP 4.1 : CREATING RECURSIVE SPLITTER
Recursive Splitter Created Successfully!

STEP 4.2 : GENERATING RECURSIVE CHUNKS

RECURSIVE CHUNK SUMMARY
Total Chunks Created : 6

Chunk  1 | Characters :  435 | Words :   54
Chunk  2 | Characters :  479 | Words :   61
Chunk  3 | Characters :  481 | Words :   67
Chunk  4 | Characters :  482 | Words :   71
Chunk  5 | Characters :  469 | Words :   77
Chunk  6 | Characters :  234 | Words :   39

CHUNK 1
Characters : 435
Words      : 54

Chunk Text
----------------------------------------------------------------------------------------------------
Details
Address
New York, NY
United States
Phone
917-678-5454
Email
kaitlyn_rianalyst@gmail.com
Skills
Business Development and
Management
Leadership
@eeee
Problem Solving
@e@eee0e
Attention to Detail
@eeee
Interpersonal Skills
@eeee
Finance and Accounting
@eeee
Languages
English
@eeeee0
Spanish; Castilian
@eee
French
@eoee
Kaitlyn Riley
Business
Profile
Dedicated Business Analyst with a strong understandin